# **[Grand Prix Name] [Year]**
## 01: Results and Lap Analysis

In [ ]:
# Setup & Imports
import sys
from pathlib import Path
sys.path.append("../..")

import Sliplib.slipload as sl
import Sliplib.selection as slipclassify
import Sliplib.GridSum as grid_sum
import Sliplib.plotting as slplot
import Sliplib.battle_finder as battle
from configuration.writer import write_contenders

In [ ]:
# Session Configuration
YEAR = 2025
GP_NAME = "Austrian Grand Prix"  # Change to target Grand Prix name

### 1. Qualifying Analysis
Load qualifying session, inspect results, and export summary tables.

In [ ]:
# Load Qualifying Session
session_Q = sl.load_gp(YEAR, GP_NAME, "Q")
session_Q.load()
q_results = session_Q.results
q_laps = session_Q.laps

In [ ]:
# Clean and Export Qualifying Results
clean_q = slipclassify.clean_result_Quali(q_results)
clean_q.to_csv("../data/Quali_Result.csv")
clean_q

In [ ]:
# Qualifying Fastest Lap
q_fastest = q_laps.pick_fastest()
fastest_summary = slipclassify.analyse_fastest(q_fastest)
fastest_summary

In [ ]:
# Stint Count & Fresh Tyre Period per Driver
lap_stint = slipclassify.get_lap_stint(q_laps)
lap_stint.to_csv("../data/Quali_Lap_stint.csv", index=False)

fresh_tyre = slipclassify.get_fresh_tyre_period(q_laps)
fresh_tyre.to_csv("../data/Fresh_Tyre_Period.csv", index=False)
fresh_tyre.head(10)

### 2. Race Results & Race Control
Analyze race classification, safety cars/flags, and position gains.

In [ ]:
# Load Race Session
session_R = sl.load_gp(YEAR, GP_NAME, "R")
session_R.load()
r_results = session_R.results
r_laps = session_R.laps

In [ ]:
# Clean and Export Race Results
clean_race = slipclassify.clean_result_race(r_results)
clean_race.to_csv("../data/Race_Result.csv", index=False)
clean_race

In [ ]:
# Track & Race Control History
track_history = slipclassify.analyse_track_History(session_R.race_control_messages)
track_history

In [ ]:
# Position Gain / Loss Calculation
pos_gain = grid_sum.calc_pos_gain(r_results)
pos_gain

### 3. Automated Battle Detection & Race Progression
Identify key rivalries and on-track battles, save contenders for telemetry analysis.

In [ ]:
# Detect Top Battles and Rival Pairs
contenders, rivals = battle.find_top_battles(r_laps, n=6)

# Save to local race data/battles.json and sync legacy config
write_contenders(contenders, rivals, local_path="../data/battles.json")
print("Contenders:", contenders)
print("Rivals:", rivals)

In [ ]:
# Plot Race Progression for Top Drivers
drivers_dict = slipclassify.select_drivers(r_laps, contenders[:6])
slplot.plot_race_progression(
    list(drivers_dict.values()),
    save_path="../plots/race_progression_top.png"
)